# Pixels to Predictions — DL Vision Challenge

Fine-tunes **SmolVLM-500M-Instruct** with LoRA (≤ 5 M trainable params) and
generates `submission.csv`.

**Primary GPU used: Kaggle Tesla T4 x2 (2 x 15 GB GPUs)** 
Considering the competition rule, the following notebook can run on Free Tier Tesla T4 x2 GPUs



## Inputs this notebook expects

Both attached as Kaggle Datasets in the right-hand panel:

1. **Competition data** at `/kaggle/input/datasets/dheerajpakala/data-set/`
   - `train.csv`, `val.csv`, `test.csv`
   - `images/images/{train,val,test}/...`
2. **SmolVLM model files** at `/kaggle/input/datasets/dheerajpakala/model-data/smolvlm-500m/`
   - `model.safetensors`, `config.json`, `tokenizer.json`, etc.



## 0. Mode switch — set this before running


## Notebook settings 

WE have provided 2 mode
* prep  → download SmolVLM (internet ON, no GPU)
* train → fine-tune + predict (internet OFF, GPU)

In [2]:

MODE = "train"          # "prep"  → download SmolVLM (internet ON, no GPU)
                         # "train" → fine-tune + predict (internet OFF, GPU)

HF_MODEL_ID          = "HuggingFaceTB/SmolVLM-500M-Instruct"
LOCAL_MODEL_DIR_NAME = "smolvlm-500m"   # folder name inside /kaggle/working

print(f"MODE = {MODE!r}")


MODE = 'train'


## 1. Phase 1 — prep mode (download the model, internet ON)

Only runs when `MODE == "prep"`. Downloads only the files we actually need
(safetensors weights + tokenizer + processor configs), giving a tidy ~1 GB
folder you can save as a Kaggle Dataset.

After this finishes:
1. **Save Version → Save & Run All (Commit)** — wait for it to finish.
2. Open the committed version's **Output** → click **"New Dataset"**.
3. Set `MODE = "train"` and re-run with GPU on, internet off.


In [3]:
if MODE == "prep":
    import os, sys, subprocess
    from pathlib import Path

    try:
        from huggingface_hub import snapshot_download
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
        from huggingface_hub import snapshot_download

    target = Path("/kaggle/working") / LOCAL_MODEL_DIR_NAME
    target.mkdir(parents=True, exist_ok=True)

    print(f"Downloading {HF_MODEL_ID} -> {target}")
    snapshot_download(
        HF_MODEL_ID,
        local_dir=str(target),
        allow_patterns=[
            "*.json",            # config / tokenizer / processor configs
            "*.safetensors",     # weights
            "*.txt",             # special_tokens_map etc.
            "*.model",           # sentencepiece tokenizer (if any)
        ],
        ignore_patterns=[
            "*.onnx", "onnx/*",  # ONNX runtime files (huge, not needed)
            "*.gguf",            # llama.cpp quantised weights
            "*.bin",             # duplicate pytorch weights
            "*.msgpack",
            "*.h5",
            "tf_model.*",
        ],
    )

    files    = sorted(p.name for p in target.iterdir() if p.is_file())
    total_mb = sum(p.stat().st_size for p in target.rglob("*") if p.is_file()) / 1e6
    print(f"\n✓ Downloaded {len(files)} files, total {total_mb:.0f} MB")
    print("Files:", files)

    print("\n" + "=" * 60)
    print("NEXT STEPS:")
    print("  1. Save Version -> Save & Run All (Commit)")
    print("  2. Open committed version's Output panel")
    print("  3. Click 'New Dataset' to turn this folder into a Kaggle Dataset")
    print("  4. Set MODE = \"train\" and re-run with GPU on, internet off")
    print("=" * 60)

    raise SystemExit(0)
else:
    print(f"Skipping prep (MODE = {MODE!r}), continuing to training pipeline.")


Skipping prep (MODE = 'train'), continuing to training pipeline.


## 2. Imports, reproducibility & GPU detection

 **Kaggle Tesla T4 x2** is the primary tuning profile


In [32]:
import os, json, random, math, gc
from pathlib import Path
import json

import numpy as np
import pandas as pd
from PIL import Image

# Helps reduce CUDA memory fragmentation on long Kaggle runs.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
from torch.utils.data import Dataset, DataLoader

# Force HF transformers offline — no accidental network calls
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"]       = "1"

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ── Detect Kaggle GPU topology ────────────────────────────────────────────
if torch.cuda.is_available():
    NUM_GPUS = torch.cuda.device_count()
    GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(NUM_GPUS)]
    PRIMARY_DEVICE_INDEX = 0
    device = torch.device(f"cuda:{PRIMARY_DEVICE_INDEX}")
    GPU_NAME = GPU_NAMES[PRIMARY_DEVICE_INDEX].lower()
    VRAM_GB = torch.cuda.get_device_properties(PRIMARY_DEVICE_INDEX).total_memory / 1e9
    TOTAL_VRAM_GB = sum(torch.cuda.get_device_properties(i).total_memory for i in range(NUM_GPUS)) / 1e9
    PRIMARY_CAP = torch.cuda.get_device_capability(PRIMARY_DEVICE_INDEX)
else:
    NUM_GPUS = 0
    GPU_NAMES = []
    PRIMARY_DEVICE_INDEX = None
    device = torch.device("cpu")
    GPU_NAME = ""
    VRAM_GB = 0
    TOTAL_VRAM_GB = 0
    PRIMARY_CAP = (0, 0)

IS_MULTI_GPU = NUM_GPUS > 1
ALL_T4       = NUM_GPUS > 0 and all("t4" in name.lower() for name in GPU_NAMES)
IS_DUAL_T4   = NUM_GPUS >= 2 and ALL_T4

# Specific GPU detection based on the primary visible Kaggle GPU.
IS_T4           = "t4" in GPU_NAME
IS_P100         = "p100" in GPU_NAME

# Kaggle T4 is the primary profile here, so T4/P100 stay on fp16.

DTYPE = torch.float16
DTYPE_NAME = "float16"

# Base GPU class uses per-GPU memory; Kaggle dual T4 is the primary tuning target.

if IS_P100 or IS_T4 or VRAM_GB >= 14:
    GPU_CLASS = "small"      # T4/P100 16 GB class

GPU_TUNING_PROFILE = "dual_t4" if IS_DUAL_T4 else GPU_CLASS

print(f"Device:          {device}")
print(f"CUDA devices:    {NUM_GPUS}")
if torch.cuda.is_available():
    for index, name in enumerate(GPU_NAMES):
        print(f"  GPU {index}: {name}")
    print(f"Primary GPU VRAM: {VRAM_GB:.1f} GB")
    print(f"Total VRAM:      {TOTAL_VRAM_GB:.1f} GB")
    print(f"Compute:         {PRIMARY_CAP[0]}.{PRIMARY_CAP[1]}")
print(f"Dtype:           {DTYPE_NAME}")
print(f"GPU class:       {GPU_CLASS}")
print(f"Tuning profile:  {GPU_TUNING_PROFILE}")

Device:          cuda:0
CUDA devices:    2
  GPU 0: Tesla T4
  GPU 1: Tesla T4
Primary GPU VRAM: 15.6 GB
Total VRAM:      31.3 GB
Compute:         7.5
Dtype:           float16
GPU class:       small
Tuning profile:  dual_t4


## 3. Configuring Data and Model 

In [41]:
from pathlib import Path
import json

# ── Model files: uploaded Kaggle Model/Dataset ───────────────────────────
MODEL_ROOT = Path("/kaggle/input/models/nidhish1010/huggingfacetbsmolvlm-500m-instruct/pytorch/default/1")

# If files are directly inside MODEL_ROOT, use it.
# If files are inside smolvlm-500m-instruct/, use that.
if (MODEL_ROOT / "config.json").exists():
    MODEL_DIR = MODEL_ROOT
elif (MODEL_ROOT / "smolvlm-500m-instruct" / "config.json").exists():
    MODEL_DIR = MODEL_ROOT / "smolvlm-500m-instruct"
else:
    raise FileNotFoundError(f"Could not find config.json under {MODEL_ROOT}")

MODEL_ID_OR_PATH = str(MODEL_DIR)


# ── Competition data ─────────────────────────────────────────────────────
COMP_ROOT  = Path("/kaggle/input/datasets/dheerajpakala/data-set")
IMAGE_ROOT = COMP_ROOT / "images" / "images"

if not IMAGE_ROOT.exists():
    IMAGE_ROOT = COMP_ROOT / "images"

CSV_DIR = COMP_ROOT


# ── Auto-search fallback ─────────────────────────────────────────────────
def _autosearch(name: str):
    print(f"  searching /kaggle/input for {name!r}...")
    for p in Path("/kaggle/input").rglob(name):
        return p.parent
    return None


print("=" * 60)
print("MODEL DIR")
print("=" * 60)

if not MODEL_DIR.exists():
    found = _autosearch("model.safetensors")
    if found is None:
        raise FileNotFoundError(
            "Could not locate model.safetensors under /kaggle/input."
        )
    MODEL_DIR = found
    MODEL_ID_OR_PATH = str(MODEL_DIR)

config_path = MODEL_DIR / "config.json"

with open(config_path, "r") as f:
    config = json.load(f)

name = config.get("_name_or_path", "SmolVLM-500M-Instruct")
arch = config.get("architectures", ["Unknown"])[0]
dtype = config.get("torch_dtype", config.get("dtype", "N/A"))
layers = config.get("num_hidden_layers", "N/A")

print(f"Model Name: {name}")
print(f"Architecture: {arch}")
print(f"Data Type: {dtype}")
print(f"Layers: {layers}")

files = sorted(f.name for f in MODEL_DIR.iterdir() if f.is_file())
size_mb = sum(f.stat().st_size for f in MODEL_DIR.iterdir() if f.is_file()) / 1e6

print(f"  ✓ {MODEL_DIR}")
print(f"    {len(files)} files, {size_mb:.0f} MB")


assert any(f.endswith(".safetensors") or f.endswith(".bin") for f in files), "no weights file"
assert "config.json" in files, "no config.json"


print("\n" + "=" * 60)
print("COMPETITION DATA")
print("=" * 60)

if not (CSV_DIR / "train.csv").exists():
    print(f"  ✗ train.csv not at: {CSV_DIR}")
    found = _autosearch("train.csv")

    if found is None:
        raise FileNotFoundError("Could not locate train.csv under /kaggle/input.")

    COMP_ROOT = found
    CSV_DIR = found
    IMAGE_ROOT = COMP_ROOT / "images" / "images"

    if not IMAGE_ROOT.exists():
        IMAGE_ROOT = COMP_ROOT / "images"

    print(f"  ✓ Auto-found at: {COMP_ROOT}")

print(f"  COMP_ROOT:  {COMP_ROOT}")
print(f"  CSV_DIR:    {CSV_DIR}")
print(f"  IMAGE_ROOT: {IMAGE_ROOT}  (exists={IMAGE_ROOT.exists()})")

for split in ("train", "val", "test"):
    split_dir = IMAGE_ROOT / split
    n = len(list(split_dir.glob("*"))) if split_dir.exists() else 0
    mark = "✓" if n > 0 else "✗"
    print(f"    {mark} {split}: {n} images")

for csv in ("train.csv", "val.csv", "test.csv"):
    p = CSV_DIR / csv
    print(f"    {'✓' if p.exists() else '✗'} {p.name}")

print("\nMODEL_ID_OR_PATH =", MODEL_ID_OR_PATH)

MODEL DIR
Model Name: SmolVLM-500M-Instruct
Architecture: Idefics3ForConditionalGeneration
Data Type: bfloat16
Layers: N/A
  ✓ /kaggle/input/models/nidhish1010/huggingfacetbsmolvlm-500m-instruct/pytorch/default/1/smolvlm-500m-instruct
    14 files, 1020 MB

COMPETITION DATA
  ✗ train.csv not at: /kaggle/input/datasets/dheerajpakala/data-set
  searching /kaggle/input for 'train.csv'...
  ✓ Auto-found at: /kaggle/input/competitions/pixels-to-predictions
  COMP_ROOT:  /kaggle/input/competitions/pixels-to-predictions
  CSV_DIR:    /kaggle/input/competitions/pixels-to-predictions
  IMAGE_ROOT: /kaggle/input/competitions/pixels-to-predictions/images/images  (exists=True)
    ✓ train: 3109 images
    ✓ val: 1048 images
    ✓ test: 1008 images
    ✓ train.csv
    ✓ val.csv
    ✓ test.csv

MODEL_ID_OR_PATH = /kaggle/input/models/nidhish1010/huggingfacetbsmolvlm-500m-instruct/pytorch/default/1/smolvlm-500m-instruct


## 4. Load CSVs and build prompts

In [42]:
train_df = pd.read_csv(CSV_DIR / "train.csv")
val_df   = pd.read_csv(CSV_DIR / "val.csv")
test_df  = pd.read_csv(CSV_DIR / "test.csv")

# choices is a JSON-encoded list — parse it
for df in (train_df, val_df, test_df):
    df["choices"] = df["choices"].apply(json.loads)

print(f"Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}")
print("Columns:", train_df.columns.tolist())
train_df.head(2)


Train: 3,109  |  Val: 1,048  |  Test: 1,008
Columns: ['id', 'image_path', 'question', 'choices', 'num_choices', 'answer', 'hint', 'lecture', 'solution', 'task', 'grade', 'subject', 'topic', 'category', 'skill']


,id,image_path,question,choices,num_choices,answer,hint,lecture,solution,task,grade,subject,topic,category,skill
0,train_07667,images/train/train_07667.png,Why might putting each tadpole in its own pool...,[the male's tadpoles will be larger when they ...,3,2,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...
1,train_02628,images/train/train_02628.png,Why might forming strong social bonds with oth...,"[the female's offspring will live longer, the ...",3,0,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...


In [43]:
CHOICE_LETTERS = "ABCDEFGHIJ"


def get_image_path(rel_path: str) -> Path:
    """CSVs store paths like 'images/val/val_00003.png'. Strip the leading 'images/'
    because IMAGE_ROOT already points at the images folder."""
    if rel_path.startswith("images/"):
        rel_path = rel_path[len("images/"):]
    return IMAGE_ROOT / rel_path


def build_prompt_text(row) -> str:
    """The textual part of the prompt (no <image> token — the chat template adds it)."""
    parts = []
    for col in ("lecture", "hint"):
        val = row.get(col, "")
        if pd.notna(val) and str(val).strip():
            parts.append(str(val).strip())
    context_str = "\n".join(parts)

    choices = row["choices"]
    choices_str = "\n".join(f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))

    p = ""
    if context_str:
        p += f"Context:\n{context_str}\n\n"
    p += f"Question: {row['question']}\n"
    p += f"Choices:\n{choices_str}\n"
    p += "Answer:"
    return p


def build_chat_prompt(row) -> str:
    """Apply SmolVLM's chat template, which inserts the proper <image> tokens."""
    user_text = build_prompt_text(row)
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": user_text},
            ],
        }
    ]
    return processor.apply_chat_template(messages, add_generation_prompt=True)


# Sanity check — needs `processor` to exist, so move this AFTER section 5
# (commented out for now; the dataset cell uses build_chat_prompt directly)
# print(build_chat_prompt(train_df.iloc[0])[:500])

## 5. Load SmolVLM (offline, auto-dtype)

Loads in **float16** on Kaggle Tesla T4 / P100, which is the primary Kaggle
profile here. 


In [44]:
from transformers import AutoProcessor
from transformers.modeling_utils import PreTrainedModel


def _iter_transformer_tensors(module):
    for param in module.parameters():
        yield param

    for buffer in module.buffers():
        yield buffer

    for submodule in module.modules():
        former_parameters = getattr(submodule, "_former_parameters", None)
        if not former_parameters:
            continue
        for tensor in former_parameters.values():
            if tensor is not None:
                yield tensor


def _safe_transformers_device(module):
    for tensor in _iter_transformer_tensors(module):
        return tensor.device

    raise RuntimeError(
        f"Could not infer device for {type(module).__name__}; no parameters, buffers, or DataParallel replica tensors found."
    )


def _safe_transformers_dtype(module):
    for tensor in _iter_transformer_tensors(module):
        if torch.is_floating_point(tensor):
            return tensor.dtype

    for tensor in _iter_transformer_tensors(module):
        return tensor.dtype

    raise RuntimeError(
        f"Could not infer dtype for {type(module).__name__}; no parameters, buffers, or DataParallel replica tensors found."
    )


PreTrainedModel.device = property(_safe_transformers_device)
PreTrainedModel.dtype = property(_safe_transformers_dtype)

try:
    # transformers >= 5.0 (Kaggle's current image, Jan 2026+)
    from transformers import AutoModelForImageTextToText as AutoModelForVision2Seq
    print("Using AutoModelForImageTextToText (transformers >= 5.0)")
except ImportError:
    # transformers < 5.0 fallback
    from transformers import AutoModelForVision2Seq
    print("Using AutoModelForVision2Seq (transformers < 5.0)")

processor = AutoProcessor.from_pretrained(MODEL_ID_OR_PATH, trust_remote_code=True)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

# On big GPUs we can skip low_cpu_mem_usage (faster load)
load_kwargs = dict(
    torch_dtype=DTYPE,
    trust_remote_code=True,
 )
if GPU_CLASS in ("small"):
    load_kwargs["low_cpu_mem_usage"] = True

model = AutoModelForVision2Seq.from_pretrained(MODEL_ID_OR_PATH, **load_kwargs).to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"✓ Loaded {type(model).__name__}  ({n_params/1e6:.1f}M params, dtype={DTYPE_NAME})")
print("✓ Installed safe PreTrainedModel.device and .dtype fallbacks for DataParallel replicas")

if torch.cuda.is_available():
    used_gb = torch.cuda.memory_allocated(0) / 1e9
    print(f"  GPU mem allocated after load: {used_gb:.1f} / {VRAM_GB:.1f} GB")


Using AutoModelForImageTextToText (transformers >= 5.0)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

✓ Loaded Idefics3ForConditionalGeneration  (507.5M params, dtype=float16)
✓ Installed safe PreTrainedModel.device and .dtype fallbacks for DataParallel replicas
  GPU mem allocated after load: 1.0 / 15.6 GB


## 6. Per-letter token ids

To score multiple-choice answers we read the logit of the **letter token**
(` A`, ` B`, …) right after `Answer:`. Pre-compute the ids.


In [45]:
def letter_token_id(letter: str) -> int:
    """Token id of ' A', ' B', ... (with leading space)."""
    ids = processor.tokenizer.encode(f" {letter}", add_special_tokens=False)
    return ids[0]

LETTER_TOKEN_IDS = [letter_token_id(L) for L in CHOICE_LETTERS]
print("Letter -> token id:")
for L, tid in zip(CHOICE_LETTERS, LETTER_TOKEN_IDS):
    print(f"  {L!r:>3} -> {tid:>6}  ({processor.tokenizer.decode([tid])!r})")


Letter -> token id:
  'A' ->    330  (' A')
  'B' ->    389  (' B')
  'C' ->    340  (' C')
  'D' ->    422  (' D')
  'E' ->    414  (' E')
  'F' ->    426  (' F')
  'G' ->    452  (' G')
  'H' ->    407  (' H')
  'I' ->    339  (' I')
  'J' ->    530  (' J')


## 7. PyTorch dataset

Image resolution scales with GPU class — bigger GPU = higher res = better
chart/diagram understanding.


In [47]:
# Auto-scale training image resolution
IMG_SIZE = {

    "dual_t4": 760,
    "small":   480,
}[GPU_TUNING_PROFILE]
print(f"TRAIN_IMAGE_EDGE = {IMG_SIZE} (tuning profile: {GPU_TUNING_PROFILE})")


class ScienceQADataset(Dataset):
    def __init__(self, df: pd.DataFrame, is_train: bool = True):
        self.df = df.reset_index(drop=True)
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def _load_img(self, rel_path: str) -> Image.Image:
        img = Image.open(get_image_path(rel_path)).convert("RGB")
        img.thumbnail((IMG_SIZE, IMG_SIZE), Image.BICUBIC)
        return img

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        out = {
            "image":   self._load_img(row["image_path"]),
            "prompt":  build_chat_prompt(row),
            "choices": row["choices"],
            "id":      row["id"],
            "question": row["question"],
            "lecture":  row.get("lecture", ""),
            "hint":     row.get("hint", ""),
        }
        if "answer" in row and pd.notna(row.get("answer")):
            out["answer"] = int(row["answer"])
        return out

train_ds = ScienceQADataset(train_df, is_train=True)
val_ds   = ScienceQADataset(val_df,   is_train=False)
test_ds  = ScienceQADataset(test_df,  is_train=False)
print(f"Datasets ready: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")


TRAIN_IMAGE_EDGE = 760 (tuning profile: dual_t4)
Datasets ready: train=3109, val=1048, test=1008


## 8. Attach LoRA adapters (≤ 5 M trainable params)

Targets attention projection matrices. The loop tries ranks from large to
small and stops at the first one that fits the **5 000 000** competition cap.


In [48]:
from peft import LoraConfig, get_peft_model, TaskType

MAX_TRAINABLE_PARAMS = 5_000_000
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
]


def trainable_count(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


selected_rank = None
for rank in (32, 24, 16, 12, 8, 4):
    cfg = LoraConfig(
        r=rank,
        lora_alpha=rank * 2,
        target_modules=TARGET_MODULES,
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    test_model = get_peft_model(model, cfg)
    n = trainable_count(test_model)
    fits = n <= MAX_TRAINABLE_PARAMS
    print(f"  rank={rank:>2}  trainable={n:>10,}  ", "✓ fits" if fits else "✗ too big")
    if fits:
        selected_rank = rank
        model = test_model
        break
    # undo to try smaller rank
    model = test_model.unload() if hasattr(test_model, "unload") else test_model.base_model.model

assert selected_rank is not None, "Could not fit LoRA under 5M params — reduce target_modules"
print(f"\n✓ Using LoRA rank {selected_rank}  →  {trainable_count(model):,} trainable params (cap 5,000,000)")
model.print_trainable_parameters()


  rank=32  trainable= 8,323,072   ✗ too big


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


  rank=24  trainable= 6,242,304   ✗ too big
  rank=16  trainable= 4,161,536   ✓ fits

✓ Using LoRA rank 16  →  4,161,536 trainable params (cap 5,000,000)
trainable params: 4,161,536 || all params: 511,643,840 || trainable%: 0.8134


## 9. Training — supervised fine-tuning on the answer letter

For each training example we build the prompt ending in `Answer:`, append the
correct letter (e.g. ` B`), and use causal-LM loss with **labels masked to -100
everywhere except on that one letter token**.

Hyperparameters auto-scale with the GPU class.


In [49]:
def make_training_inputs(images, prompts, answer_letters):
    full_texts = [p + f" {a}" for p, a in zip(prompts, answer_letters)]
    enc = processor(
        text=full_texts,
        images=images,
        return_tensors="pt",
        padding=True,
    )
    #enc = {k: v.to(device) if torch.is_tensor(v) else v for k, v in enc.items()}

    input_ids      = enc["input_ids"]
    attention_mask = enc["attention_mask"]
    labels         = torch.full_like(input_ids, -100)

    # Supervise only the final non-pad token (= the answer letter)
    seq_lens = attention_mask.sum(dim=1)
    for i, L in enumerate(seq_lens):
        labels[i, L - 1] = input_ids[i, L - 1]

    enc["labels"] = labels
    return enc


def train_collate(batch):
    images, prompts, letters = [], [], []
    for b in batch:
        n = len(b["choices"])
        perm = list(range(n))
        random.shuffle(perm)
        perm_choices = [b["choices"][p] for p in perm]
        new_answer_pos = perm.index(b["answer"])

        # ── inlined _build_prompt_with_choices ──────────────────────────
        parts = []
        for col in ("lecture", "hint"):
            val = b.get(col, "")
            if pd.notna(val) and str(val).strip():
                parts.append(str(val).strip())
        context_str = "\n".join(parts)
        choices_str = "\n".join(f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(perm_choices))

        user_text = ""
        if context_str:
            user_text += f"Context:\n{context_str}\n\n"
        user_text += f"Question: {b['question']}\n"
        user_text += f"Choices:\n{choices_str}\n"
        user_text += "Answer:"

        messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": user_text}]}]
        prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
        # ────────────────────────────────────────────────────────────────

        images.append(b["image"])
        prompts.append(prompt)
        letters.append(CHOICE_LETTERS[new_answer_pos])

    return make_training_inputs(images, prompts, letters)


In [50]:
from torch.utils.data import WeightedRandomSampler

# Keep validation held out: the sampler must follow train_ds and train_df only.
all_labels = train_df["answer"].astype(int).tolist()
class_counts = np.bincount(all_labels)
weights = 1.0 / class_counts[np.array(all_labels)]
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)


In [52]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from tqdm.auto import tqdm


def unwrap_model(model):
    return model.module if isinstance(model, torch.nn.DataParallel) else model


def configure_training_memory(model):
    core_model = unwrap_model(model)

    if hasattr(core_model, "config") and hasattr(core_model.config, "use_cache"):
        core_model.config.use_cache = False

    if GPU_TUNING_PROFILE not in ("dual_t4", "small"):
        return False

    gc_target = core_model if hasattr(core_model, "gradient_checkpointing_enable") else getattr(core_model, "base_model", None)
    if gc_target is None or not hasattr(gc_target, "gradient_checkpointing_enable"):
        return False

    try:
        gc_target.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    except TypeError:
        gc_target.gradient_checkpointing_enable()

    input_grad_target = core_model if hasattr(core_model, "enable_input_require_grads") else gc_target
    if hasattr(input_grad_target, "enable_input_require_grads"):
        input_grad_target.enable_input_require_grads()

    return True


def maybe_enable_data_parallel(model, batch_size):
    if not IS_MULTI_GPU:
        return model, False

    if batch_size < NUM_GPUS:
        print(f"Multi-GPU available but GLOBAL_BATCH_SIZE={batch_size} < NUM_GPUS={NUM_GPUS}; training will stay on one GPU.")
        return model, False

    wrapped_model = torch.nn.DataParallel(model, device_ids=list(range(NUM_GPUS)))
    return wrapped_model, True


model = unwrap_model(model)
MEMORY_SAVER_MODE = configure_training_memory(model)

# ── Hyperparameters auto-scale with the tuning profile ────────────────────
# Kaggle dual T4 is the primary profile for this notebook.
HP = {
    "dual_t4": dict(NUM_EPOCHS=3, BATCH_SIZE=4, GRAD_ACCUM=8,  LR=1e-4, NUM_WORKERS=2),
    "small":   dict(NUM_EPOCHS=1, BATCH_SIZE=1, GRAD_ACCUM=16, LR=2e-4, NUM_WORKERS=1),
}[GPU_TUNING_PROFILE]

NUM_EPOCHS        = HP["NUM_EPOCHS"]
BATCH_SIZE        = HP["BATCH_SIZE"]
GRAD_ACCUM        = HP["GRAD_ACCUM"]
LEARNING_RATE     = HP["LR"]
TRAIN_NUM_WORKERS = HP["NUM_WORKERS"]
WEIGHT_DECAY      = 0.01
WARMUP_RATIO      = 0.05
MAX_TRAIN_STEPS   = None     # set to e.g. 300 for a quick smoke test

model, USING_DATA_PARALLEL = maybe_enable_data_parallel(model, BATCH_SIZE)
ACTIVE_GPUS = NUM_GPUS if USING_DATA_PARALLEL else (1 if torch.cuda.is_available() else 0)
PER_GPU_BATCH = max(1, math.ceil(BATCH_SIZE / max(1, ACTIVE_GPUS)))
eff_batch = BATCH_SIZE * GRAD_ACCUM

print(f"Hyperparameters for tuning profile {GPU_TUNING_PROFILE!r}:")
print(f"  NUM_GPUS          = {NUM_GPUS}")
print(f"  MULTI_GPU_MODE    = {'data_parallel' if USING_DATA_PARALLEL else 'single_gpu'}")
print(f"  NUM_EPOCHS        = {NUM_EPOCHS}")
print(f"  GLOBAL_BATCH_SIZE = {BATCH_SIZE}")
print(f"  PER_GPU_BATCH     = {PER_GPU_BATCH}")
print(f"  GRAD_ACCUM        = {GRAD_ACCUM}  (effective batch {eff_batch})")
print(f"  LEARNING_RATE     = {LEARNING_RATE}")
print(f"  WEIGHT_DECAY      = {WEIGHT_DECAY}")
print(f"  WARMUP_RATIO      = {WARMUP_RATIO}")
print(f"  TRAIN_IMAGE_EDGE  = {IMG_SIZE}")
print(f"  TRAIN_NUM_WORKERS = {TRAIN_NUM_WORKERS}")
print(f"  MEMORY_SAVER_MODE = {MEMORY_SAVER_MODE}")

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=train_collate,
    num_workers=TRAIN_NUM_WORKERS,
    pin_memory=True,
    sampler=sampler,
)

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_steps     = steps_per_epoch * NUM_EPOCHS
if MAX_TRAIN_STEPS is not None:
    total_steps = min(total_steps, MAX_TRAIN_STEPS)
warmup_steps = max(1, int(total_steps * WARMUP_RATIO))


def cosine_with_warmup(step):
    if step < warmup_steps:
        return step / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1 + math.cos(math.pi * progress))


scheduler = LambdaLR(optimizer, cosine_with_warmup)
print(f"\nTotal optimiser steps: {total_steps}  (warmup {warmup_steps})")

if torch.cuda.is_available():
    torch.cuda.empty_cache()

Hyperparameters for tuning profile 'dual_t4':
  NUM_GPUS          = 2
  MULTI_GPU_MODE    = data_parallel
  NUM_EPOCHS        = 3
  GLOBAL_BATCH_SIZE = 4
  PER_GPU_BATCH     = 2
  GRAD_ACCUM        = 8  (effective batch 32)
  LEARNING_RATE     = 0.0001
  WEIGHT_DECAY      = 0.01
  WARMUP_RATIO      = 0.05
  TRAIN_IMAGE_EDGE  = 760
  TRAIN_NUM_WORKERS = 2
  MEMORY_SAVER_MODE = True

Total optimiser steps: 294  (warmup 14)


In [53]:
model.train()
global_step  = 0
running_loss = 0.0
log_every    = 20

autocast_ctx = torch.amp.autocast(device_type="cuda", dtype=DTYPE) if torch.cuda.is_available() else torch.amp.autocast(device_type="cpu")

pbar = tqdm(total=total_steps, desc="train")
for epoch in range(NUM_EPOCHS):
    optimizer.zero_grad()
    for i, batch in enumerate(train_loader):
        batch = {k: v.to(device, non_blocking=True) if torch.is_tensor(v) else v
                 for k, v in batch.items()}
        is_last_batch = (i + 1) == len(train_loader)
        with autocast_ctx:
            outputs = model(**batch)
            raw_loss = outputs.loss
            if raw_loss.ndim > 0:
                raw_loss = raw_loss.mean()
            loss = raw_loss / GRAD_ACCUM
        loss.backward()
        running_loss += loss.item() * GRAD_ACCUM

        if (i + 1) % GRAD_ACCUM == 0 or is_last_batch:
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1
            pbar.update(1)

            if global_step % log_every == 0:
                avg = running_loss / (log_every * GRAD_ACCUM)
                pbar.set_postfix(loss=f"{avg:.4f}", lr=f"{scheduler.get_last_lr()[0]:.2e}")
                running_loss = 0.0

            if global_step >= total_steps:
                break
    if global_step >= total_steps:
        break
pbar.close()

ADAPTER_DIR = "/kaggle/working/lora_adapter" if Path("/kaggle/working").exists() else "lora_adapter"
save_model = model.module if isinstance(model, torch.nn.DataParallel) else model
save_model.save_pretrained(ADAPTER_DIR)
print(f"✓ LoRA adapter saved to {ADAPTER_DIR}")

# Free a bit of memory before inference
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


train:   0%|          | 0/294 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


✓ LoRA adapter saved to /kaggle/working/lora_adapter


## 10. Inference — plain + cyclic-permutation TTA

Two prediction modes:
- **Plain**: standard log-likelihood pick over original choice order
- **Cyclic-TTA**: for each test example with N choices, runs N inference passes,
  each with the choices cyclically rotated by k positions. Predictions are
  un-rotated back to original choice positions and averaged. Cancels position bias.

### Why this helps

VLMs have measurable position bias — a tendency to prefer specific letter
positions (often A or B) regardless of content. Cyclic permutation places
each original choice in each position exactly once, and averaging the
re-aligned probabilities cancels that bias.

In [54]:
import torch

@torch.no_grad()
def predict_batch_logits(images, prompts, choices_list):
    """Return per-example logits over valid answer-letter positions."""
    enc = processor(text=prompts, images=images, return_tensors="pt", padding=True)
    enc = {k: v.to(device, non_blocking=True) if torch.is_tensor(v) else v
           for k, v in enc.items()}

    # Use autocast to match model dtype (fp16) with input tensors
    with torch.amp.autocast(device_type="cuda", dtype=DTYPE):
        out = model(**enc)

    logits = out.logits  # (B, T, V)

    seq_lens = enc["attention_mask"].sum(dim=1) - 1
    last_logits = logits[torch.arange(logits.size(0)), seq_lens, :]  # (B, V)

    out_logits = []
    for i, choices in enumerate(choices_list):
        n = len(choices)
        valid_ids = LETTER_TOKEN_IDS[:n]
        out_logits.append(last_logits[i, valid_ids].float().cpu().numpy())
    return out_logits


def _load_rgb_image(rel_path: str) -> Image.Image:
    # Use a context manager to avoid leaking file handles during long inference runs.
    with Image.open(get_image_path(rel_path)) as img:
        return img.convert("RGB")


def _build_prompt_with_choices(row, perm_choices):
    """Same as build_chat_prompt but uses perm_choices in the given order."""
    parts = []
    for col in ("lecture", "hint"):
        val = row.get(col, "")
        if pd.notna(val) and str(val).strip():
            parts.append(str(val).strip())
    context_str = "\n".join(parts)
    choices_str = "\n".join(f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(perm_choices))

    user_text = ""
    if context_str:
        user_text += f"Context:\n{context_str}\n\n"
    user_text += f"Question: {row['question']}\n"
    user_text += f"Choices:\n{choices_str}\n"
    user_text += "Answer:"

    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": user_text}]}]
    return processor.apply_chat_template(messages, add_generation_prompt=True)


# Inference batch size auto-scales with the tuning profile
INFER_BS = {
    "dual_t4": 8,
    "small": 4,
}[GPU_TUNING_PROFILE]
print(f"Inference batch size: {INFER_BS} (global across {max(1, NUM_GPUS)} GPU(s))")


def predict_dataframe_plain(df, batch_size: int = INFER_BS, desc: str = "plain"):
    """Plain inference baseline."""
    model.eval()
    out_ids, out_preds = [], []
    rows = df.to_dict("records")

    for start in tqdm(range(0, len(rows), batch_size), desc=desc):
        chunk = rows[start:start + batch_size]
        images = [_load_rgb_image(r["image_path"]) for r in chunk]
        prompts = [build_chat_prompt(r) for r in chunk]
        choices = [r["choices"] for r in chunk]
        logits_list = predict_batch_logits(images, prompts, choices)

        for r, lg in zip(chunk, logits_list):
            e = np.exp(lg - lg.max())
            p = e / e.sum()
            out_ids.append(r["id"])
            out_preds.append(int(p.argmax()))

    return out_ids, out_preds


def predict_dataframe_tta(df, batch_size: int = INFER_BS, desc: str = "tta"):
    """Cyclic-permutation TTA with confidence weighting."""
    model.eval()
    rows = df.to_dict("records")

    rows_by_n = {}
    for r in rows:
        rows_by_n.setdefault(len(r["choices"]), []).append(r)

    weighted_probs = {}  # id -> n-vector (sum of confidence * prob)
    total_weights = {}   # id -> scalar (sum of confidences)

    for n, rows_n in rows_by_n.items():
        image_cache = {}
        for r in rows_n:
            rid = r["id"]
            weighted_probs[rid] = np.zeros(n, dtype=np.float64)
            total_weights[rid] = 0.0

        for k in range(n):
            inner_desc = f"{desc} n={n} shift={k}"
            for start in tqdm(range(0, len(rows_n), batch_size), desc=inner_desc, leave=False):
                chunk = rows_n[start:start + batch_size]

                images = []
                for r in chunk:
                    rid = r["id"]
                    if rid not in image_cache:
                        image_cache[rid] = _load_rgb_image(r["image_path"])
                    images.append(image_cache[rid])

                perm = [(i + k) % n for i in range(n)]
                prompts, choices_perm_list = [], []

                for r in chunk:
                    perm_choices = [r["choices"][p] for p in perm]
                    prompts.append(_build_prompt_with_choices(r, perm_choices))
                    choices_perm_list.append(perm_choices)

                logits_list = predict_batch_logits(images, prompts, choices_perm_list)

                for r, lg in zip(chunk, logits_list):
                    e = np.exp(lg - lg.max())
                    p = e / e.sum()
                    confidence = float(p.max())

                    orig_probs = np.zeros(n, dtype=np.float64)
                    for i in range(n):
                        orig_probs[perm[i]] = p[i]

                    rid = r["id"]
                    weighted_probs[rid] += orig_probs * confidence
                    total_weights[rid] += confidence

        for rid in image_cache:
            image_cache[rid].close()

        for r in rows_n:
            rid = r["id"]
            w = total_weights[rid]
            if w > 0:
                weighted_probs[rid] /= w

    out_ids, out_preds = [], []
    for r in rows:
        rid = r["id"]
        prob = weighted_probs[rid]
        out_ids.append(rid)
        out_preds.append(int(prob.argmax()))
    return out_ids, out_preds

Inference batch size: 8 (global across 2 GPU(s))


## 11. Sanity-check on validation — plain vs TTA

Compares plain and TTA val accuracy. If TTA helps, submit the TTA test predictions.
If TTA doesn't help, fall back to plain. The decision is printed at the end.

In [55]:
def _truth_in_prediction_order(df: pd.DataFrame, pred_ids):
    truth_map = df.set_index("id")["answer"]
    missing = [rid for rid in pred_ids if rid not in truth_map.index]
    if missing:
        raise KeyError(f"Missing {len(missing)} ids in validation truth.")
    return truth_map.loc[pred_ids].astype(int).to_numpy()


print("=" * 60)
print("PLAIN INFERENCE on val")
print("=" * 60)
val_ids_plain, val_preds_plain = predict_dataframe_plain(val_df, desc="val plain")
val_true_plain = _truth_in_prediction_order(val_df, val_ids_plain)
val_preds_plain_arr = np.asarray(val_preds_plain, dtype=np.int64)
val_acc_plain = float(np.mean(val_preds_plain_arr == val_true_plain))
n_plain = int(np.sum(val_preds_plain_arr == val_true_plain))
print(f"Plain val accuracy: {val_acc_plain:.4f}  ({n_plain}/{len(val_true_plain)})")

print("\n" + "=" * 60)
print("CYCLIC-TTA INFERENCE on val")
print("=" * 60)
val_ids_tta, val_preds_tta = predict_dataframe_tta(val_df, desc="val tta")
val_true_tta = _truth_in_prediction_order(val_df, val_ids_tta)
val_preds_tta_arr = np.asarray(val_preds_tta, dtype=np.int64)
val_acc_tta = float(np.mean(val_preds_tta_arr == val_true_tta))
n_tta = int(np.sum(val_preds_tta_arr == val_true_tta))
print(f"TTA val accuracy:   {val_acc_tta:.4f}  ({n_tta}/{len(val_true_tta)})")

print("\n" + "=" * 60)
print(f"DIFFERENCE (TTA - plain): {val_acc_tta - val_acc_plain:+.4f}")
print("=" * 60)
TTA_HELPS = val_acc_tta > val_acc_plain
if TTA_HELPS:
    print("TTA helps: submission_tta.csv will be used as submission.csv")
elif val_acc_tta == val_acc_plain:
    print("TTA matches plain: plain predictions will be used for submission.csv")
else:
    print("TTA is lower: plain predictions will be used for submission.csv")


PLAIN INFERENCE on val


val plain:   0%|          | 0/131 [00:00<?, ?it/s]

Plain val accuracy: 0.7662  (803/1048)

CYCLIC-TTA INFERENCE on val


val tta n=3 shift=0:   0%|          | 0/64 [00:00<?, ?it/s]

val tta n=3 shift=1:   0%|          | 0/64 [00:00<?, ?it/s]

...

val tta n=5 shift=3:   0%|          | 0/6 [00:00<?, ?it/s]

val tta n=5 shift=4:   0%|          | 0/6 [00:00<?, ?it/s]

val tta n=2 shift=0:   0%|          | 0/31 [00:00<?, ?it/s]

val tta n=2 shift=1:   0%|          | 0/31 [00:00<?, ?it/s]

TTA val accuracy:   0.7758  (813/1048)

DIFFERENCE (TTA - plain): +0.0095
TTA helps: submission_tta.csv will be used as submission.csv


In [56]:
VAL_RESULTS_TABLE = pd.DataFrame(
    [
        {
            "method": "plain",
            "correct": n_plain,
            "total": len(val_true_plain),
            "val_accuracy": val_acc_plain,
        },
        {
            "method": "tta",
            "correct": n_tta,
            "total": len(val_true_tta),
            "val_accuracy": val_acc_tta,
        },
    ]
).sort_values("val_accuracy", ascending=False).reset_index(drop=True)

print("\n" + "=" * 60)
print("VALIDATION SUMMARY")
print("=" * 60)
display(VAL_RESULTS_TABLE)
print(f"Chosen inference method for test submission: {'tta' if TTA_HELPS else 'plain'}")
print(f"Validation gap (TTA - plain): {val_acc_tta - val_acc_plain:+.4f}")



VALIDATION SUMMARY


,method,correct,total,val_accuracy
0,tta,813,1048,0.775763
1,plain,803,1048,0.766221


Chosen inference method for test submission: tta
Validation gap (TTA - plain): +0.0095


## 12. Predict the test set — write `submission.csv`

Writes both `submission_plain.csv` and `submission_tta.csv`, plus `submission.csv`
which equals whichever scored better on val.

In [57]:
# Plain inference on test
print("=" * 60)
print("PLAIN INFERENCE on test")
print("=" * 60)
test_ids_plain, test_preds_plain = predict_dataframe_plain(test_df, desc="test plain")
sub_plain = pd.DataFrame({"id": test_ids_plain, "answer": test_preds_plain})

# TTA inference on test
print("\n" + "=" * 60)
print("CYCLIC-TTA INFERENCE on test")
print("=" * 60)
test_ids_tta, test_preds_tta = predict_dataframe_tta(test_df, desc="test tta")
sub_tta = pd.DataFrame({"id": test_ids_tta, "answer": test_preds_tta})

# Validation helper
def validate_submission(sub: pd.DataFrame, name: str):
    assert list(sub.columns) == ["id", "answer"], f"{name}: wrong columns"
    assert sub["id"].is_unique, f"{name}: duplicate ids"
    assert set(sub["id"]) == set(test_df["id"]), f"{name}: id mismatch"
    assert sub["answer"].dtype.kind in "iu", f"{name}: answer must be integer"
    nc  = test_df.set_index("id")["num_choices"].to_dict()
    bad = sub[sub.apply(lambda r: not (0 <= r["answer"] < nc[r["id"]]), axis=1)]
    assert len(bad) == 0, f"{name}: {len(bad)} predictions out of range"

validate_submission(sub_plain, "plain")
validate_submission(sub_tta, "tta")

OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
sub_plain.to_csv(OUT_DIR / "submission_plain.csv", index=False)
sub_tta.to_csv(OUT_DIR / "submission_tta.csv", index=False)

# Pick whichever did better on val as the main submission.csv
if TTA_HELPS:
    sub_tta.to_csv(OUT_DIR / "submission.csv", index=False)
    chosen = "TTA"
else:
    sub_plain.to_csv(OUT_DIR / "submission.csv", index=False)
    chosen = "plain"

print(f"\n✓ Wrote submission_plain.csv  ({len(sub_plain)} rows)")
print(f"✓ Wrote submission_tta.csv    ({len(sub_tta)} rows)")
print(f"✓ submission.csv = {chosen} predictions (chosen by val accuracy)")

# Diff between plain and TTA
n_diff = (sub_plain["answer"].values != sub_tta["answer"].values).sum()
print(f"\nPlain vs TTA: {n_diff} predictions differ ({100*n_diff/len(sub_tta):.1f}%)")

print("\nPlain answer distribution:")
print(sub_plain["answer"].value_counts().sort_index())
print("\nTTA answer distribution:")
print(sub_tta["answer"].value_counts().sort_index())


PLAIN INFERENCE on test


test plain:   0%|          | 0/126 [00:00<?, ?it/s]


CYCLIC-TTA INFERENCE on test


test tta n=4 shift=0:   0%|          | 0/33 [00:00<?, ?it/s]

test tta n=4 shift=1:   0%|          | 0/33 [00:00<?, ?it/s]

...

test tta n=2 shift=1:   0%|          | 0/34 [00:00<?, ?it/s]

test tta n=3 shift=0:   0%|          | 0/55 [00:00<?, ?it/s]

test tta n=3 shift=1:   0%|          | 0/55 [00:00<?, ?it/s]

test tta n=3 shift=2:   0%|          | 0/55 [00:00<?, ?it/s]


✓ Wrote submission_plain.csv  (1008 rows)
✓ Wrote submission_tta.csv    (1008 rows)
✓ submission.csv = TTA predictions (chosen by val accuracy)

Plain vs TTA: 103 predictions differ (10.2%)

Plain answer distribution:
answer
0    373
1    352
2    210
3     69
4      4
Name: count, dtype: int64

TTA answer distribution:
answer
0    368
1    352
2    212
3     69
4      7
Name: count, dtype: int64


In [58]:
print("\n" + "=" * 60)
print("FINAL RESULT SUMMARY")
print("=" * 60)

result_files = pd.DataFrame(
    [
        {
            "file": "submission.csv",
            "path": str(OUT_DIR / "submission.csv"),
        },
        {
            "file": "submission_plain.csv",
            "path": str(OUT_DIR / "submission_plain.csv"),
        },
        {
            "file": "submission_tta.csv",
            "path": str(OUT_DIR / "submission_tta.csv"),
        },
    ]
)

print(f"Chosen submission: {chosen}")
print(f"Rows in final submission: {len(sub_plain)}")
print(f"Plain vs TTA differing predictions: {n_diff}")
display(result_files)

final_submission_preview = pd.read_csv(OUT_DIR / "submission.csv").head(10)
display(final_submission_preview)



FINAL RESULT SUMMARY
Chosen submission: TTA
Rows in final submission: 1008
Plain vs TTA differing predictions: 103


,file,path
0,submission.csv,/kaggle/working/submission.csv
1,submission_plain.csv,/kaggle/working/submission_plain.csv
2,submission_tta.csv,/kaggle/working/submission_tta.csv


,id,answer
0,test_01750,2
1,test_00128,1
2,test_02891,0
3,test_02425,1
4,test_00930,0
5,test_03725,0
6,test_00009,1
7,test_02880,1
8,test_01208,1
9,test_00619,0


# Saving the trained model

In [61]:
# ── Save trained LoRA / PEFT model after submission generation ─────────────
from pathlib import Path
import shutil
import torch

SAVE_DIR = Path("/kaggle/working/final_smolvlm_lora_model")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# If model is wrapped in DataParallel, unwrap it
model_to_save = model.module if isinstance(model, torch.nn.DataParallel) else model

# Save LoRA adapter weights
model_to_save.save_pretrained(SAVE_DIR)

# Save processor/tokenizer
processor.save_pretrained(SAVE_DIR)

# Copy submission files into same folder
for fname in ["submission.csv", "submission_plain.csv", "submission_tta.csv"]:
    src = Path("/kaggle/working") / fname
    if src.exists():
        shutil.copy(src, SAVE_DIR / fname)

# Zip for Kaggle output download
zip_path = shutil.make_archive(
    base_name="/kaggle/working/final_smolvlm_lora_model",
    format="zip",
    root_dir=SAVE_DIR
)

print("Saved LoRA model to:", SAVE_DIR)
print("Created zip:", zip_path)

Saved LoRA model to: /kaggle/working/final_smolvlm_lora_model
Created zip: /kaggle/working/final_smolvlm_lora_model.zip


## 13. Notes & tweaks

- **T4 x2 preset used here:** `NUM_EPOCHS=3`, `GLOBAL_BATCH_SIZE=8` (about `2` per GPU), `GRAD_ACCUM=8`, effective batch `32`, `LR=1e-4`, `WEIGHT_DECAY=0.01`, `WARMUP_RATIO=0.05`, `TRAIN_IMAGE_EDGE=320`, `INFER_BS=8`.
- **Why the second GPU was idle before:** the notebook loaded the model on one device only. It now uses `DataParallel` when multiple GPUs are visible and `GLOBAL_BATCH_SIZE >= NUM_GPUS`.
- **If one GPU is still idle:** restart the kernel, rerun from section 2, and keep `GLOBAL_BATCH_SIZE >= 2` on a 2-GPU T4 session.
- **OOM during training?** Lower `GLOBAL_BATCH_SIZE` to `2` and raise `GRAD_ACCUM` to `12`, or lower `TRAIN_IMAGE_EDGE` to `256`.
- **Quick test run?** Set `MAX_TRAIN_STEPS = 300` in section 9.
- **Want more accuracy?** Raise `NUM_EPOCHS` to `4` and rerun sections 9–12 if the session budget allows.
- **Param cap:** section 8 auto-selects the largest LoRA rank ≤ 5 M trainable params.

### TTA inference

- TTA inference is ~3-4× slower than plain (one pass per cyclic shift).
  For a 4-choice question it's 4× the work. Total test inference: ~6-12 min.
- If `TTA_HELPS` is False, the notebook automatically falls back to `submission_plain.csv` for the main `submission.csv`.
- For ensembling: keep both `submission_plain.csv` and `submission_tta.csv` —
  they have meaningfully different errors and combine well with your other models.

### Submission selection

- `submission.csv` — automatically the better of plain/TTA (based on val accuracy).
  Kaggle's "Submit to Competition" button picks this up.
- `submission_plain.csv` and `submission_tta.csv` — both saved separately for ensembling.

### Why TTA works (research-backed)

The 1st-place winner of a similar VLM multiple-choice competition (Perception
Test Challenge) used cyclic-permutation TTA + ensemble. Multiple papers
(PriDe, BOLD) confirm cyclic permutation as the standard technique to
neutralize position bias in multiple-choice VLMs.